## Sentiment Analysis
This is the continuation of Sentiment Analysis. Here we will discuss about ABAS.

## 7. Aspect Based Sentiment Analysis (ABAS)

#### 1️⃣ What is Aspect-Based Sentiment Analysis?

**Aspect-Based Sentiment Analysis (ABSA)** is a **fine-grained sentiment analysis** technique that doesn’t just determine the overall sentiment of a text (like “positive” or “negative”) but identifies **sentiment toward specific aspects or features** of an entity.  

**Example:** Yelp review for a restaurant:  
> “The food was amazing, but the service was slow.”  

- **Overall sentiment:** Mixed  
- **Aspect-based sentiment:**  
  - Food → Positive  
  - Service → Negative  

---

#### 2️⃣ Why ABSA is Important

- Overall sentiment analysis gives a **high-level view**, but businesses often want to know **what exactly people liked or disliked**.
- Useful for:
  - Product reviews (food, delivery, price, packaging)
  - Hotel/restaurant reviews (room, cleanliness, staff)
  - Social media monitoring (brand, customer support, product features)

---

#### 3️⃣ Key Components of ABSA

1. **Aspect Extraction**  
   - Identify the **target aspect or feature** in the text.  
   - Example: “food,” “service,” “price”

2. **Sentiment Classification**  
   - Determine the **sentiment toward each aspect**: positive, negative, neutral.  

**Example:**  
Text: “The camera quality is great but the battery life is short.”

- **Aspect extraction:** `camera quality`, `battery life`  
- **Aspect-level sentiment:**  
  - camera quality → Positive  
  - battery life → Negative

## 4️⃣ Approaches to ABSA

### a. Rule-Based Methods
- Use **dictionaries of aspect terms** + sentiment lexicons.  
- Example: if sentence contains “battery” → aspect = battery; if “short” → sentiment = negative.  
- **Pros:** Simple, interpretable.  
- **Cons:** Hard to scale, limited to known aspects.

### b. Machine Learning
- Train **classifiers** to predict sentiment for known aspects.  
- Input: `(sentence, aspect)` → Output: sentiment  
- Can use **SVM, Random Forest, LSTM**.

### c. Deep Learning / Transformers
- **State-of-the-art approach.**
- Use **transformers** (BERT, RoBERTa, mBERT, XLM-R) for:
  - **Aspect extraction** (NER-style tagging)
  - **Sentiment classification** for each aspect
- Examples:
  - **BERT-ABSA**: BERT fine-tuned for ABSA task  
  - **XLM-R ABSA**: Multilingual ABSA


#### 5️⃣ Example with Transformers

**Input:** `(sentence, aspect)`  
> Sentence: “The battery life is too short”  
> Aspect: `battery life`

**Output:** `Negative`  

In [ ]:
# Connect to Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#Load dataset
import pandas as pd
df=pd.read_csv('/content/drive/MyDrive/edurekaai/_data/synthetic_absa_dataset.csv')
df.head()

In [ ]:
print(df['aspect'].value_counts())

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.countplot(x='sentiment',data=df,hue='aspect')
plt.title("Aspect wise sentiment distribution")
plt.show()

In [ ]:
#Prepare Text Pairs (Aspect:Sentiments) for the models Input
df['input_pair']=df.apply(lambda x: f"What is the sentiment about the {x['aspect']} ? [SEP] {x['review']}",axis=1)
df[['input_pair','sentiment']].head()
print(df['input_pair'])

In [ ]:
#Encode Sentiment Labels / Targets
from sklearn.preprocessing import LabelEncoder

le=LabelEncoder()
df['sentiment']=le.fit_transform(df['sentiment'])
df[['input_pair','sentiment']].head()

In [ ]:
#Split Data for training and testing
from sklearn.model_selection import train_test_split

train_texts,test_texts,train_labels,test_labels=train_test_split(df['input_pair'],df['sentiment'],test_size=0.2,random_state=42)

In [ ]:
#Lets use Huggingface Transformer: for encoding the input features

from transformers import BertTokenizer
tokenizer=BertTokenizer.from_pretrained('bert-base-uncased')

train_encodings=tokenizer(list(train_texts),truncation=True,padding=True,max_length=64)
test_encodings=tokenizer(list(test_texts),truncation=True,padding=True,max_length=64)

In [ ]:
print(train_encodings)

In [ ]:
#class to create dataset in the dezired format

import torch

class ABSADataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {key: torch.tensor(val[idx]) for key, val in self.encodings.items()} | {'labels': torch.tensor(self.labels[idx])}

    def __len__(self):
        return len(self.labels)

train_dataset = ABSADataset(train_encodings, train_labels.tolist())
test_dataset = ABSADataset(test_encodings, test_labels.tolist())

In [ ]:
print(list(train_dataset))

In [ ]:
#Train BERT model with ABSA dataset

from transformers import BertForSequenceClassification, Trainer, TrainingArguments

model=BertForSequenceClassification.from_pretrained('bert-base-uncased',num_labels=3)

training_args=TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    eval_strategy='epoch',
    logging_dir="./logs",
)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

trainer.train()

In [ ]:
#Predict the sentiments
def pred_aspect_sentiment(aspect,sentence):
  input_text=f"What is the sentiment about the {aspect} ? [SEP] {sentence}"
  inputs=tokenizer(input_text,return_tensors='pt',truncation=True,padding=True)
  outputs=model(**inputs)
  pred=torch.argmax(outputs.logits,dim=1).item()
  return le.inverse_transform([pred])[0]

In [ ]:
print(pred_aspect_sentiment("battery",'The batteri life is extremely wonderful'))

### Using HuggingFace Transformers

In [ ]:
from transformers import pipeline

classifier=pipeline('text-classification',model='yangheng/deberta-v3-base-absa-v1.1')

#sample_text="The battery life is amazing, but the screen is poor"
sample_text="The battery life is ok, but the screen is poor"
#"The battery life is amazing, but the screen is poor[SEP]screen"  text pair
result_screen=classifier(sample_text,text_pair='screen')
result_battery=classifier(sample_text,text_pair='battery')

print(f"Aspect:screen sentiment:",result_screen)
print(f"Aspect:battery sentiment:",result_battery)

In [ ]:
#Fine tuning HF model

from transformers import TFAutoModelForSequenceClassification, AutoTokenizer
import tensorflow as tf

model_name = "yangheng/deberta-v3-base-absa-v1.1"

# Load tokenizer and TensorFlow model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = TFAutoModelForSequenceClassification.from_pretrained(model_name, from_pt=True)

# Prepare input
sentence = "The battery life is amazing, but the screen is too dim."
aspect = "battery life"
inputs = tokenizer(f"{sentence} [SEP] {aspect}", return_tensors="tf")

# Predict
outputs = model(**inputs)
probs = tf.nn.softmax(outputs.logits, axis=1)
pred_class = tf.argmax(probs, axis=1).numpy()[0]

labels = ["Negative", "Neutral", "Positive"]
print(f"Aspect: {aspect} | Sentiment: {labels[pred_class]} | Probabilities: {probs.numpy()[0]}")